In [1]:
import sys
from pathlib import Path

# Add project root to Python path
ROOT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

import importlib
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor
import src.evaluation as ev
importlib.reload(ev)
from src.ingest import load_machine_data, build_index

GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
RAG_ANSWER_PATH = ROOT_DIR / "data"
RAG_EVALUATE_PATH = ROOT_DIR / "data"

%load_ext autoreload
%autoreload 2

# 1. RAG Answer Generation

In [2]:
# Load ground_truth file
loaded_ground_truth = pd.read_csv(GROUND_TRUTH_OUTPUT_PATH / "ground_truth.csv")
dic_ground_truth = loaded_ground_truth.to_dict(orient='records')

# set test query
q = dic_ground_truth[1]
q

{'question': 'Is there a typical time window where tool wear failure becomes likely during continuous use?',
 'query_section': 'Tool Wear Failure',
 'document': '299b2d2994'}

In [3]:
# load machine data and get document index
documents = load_machine_data()
index = build_index(documents)

doc_idx = {}
for doc in documents:
    doc_idx[doc['id']] = doc

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71


## 1.1 Preparing RAG Answer File 1

In [4]:
openai_client = OpenAI()
instruction1 = '''
You are a predictive maintenance assistant for industrial equipment.
Answer the question using ONLY the context below. If the context doesn't
contain enough information, say so rather than guessing.
'''
ptemplate1 = '''
QUESTION: {question}

CONTEXT:
{context}

Give a clear, structured answer: what's happening, why (cite the relevant
mechanism from the context), and recommended action.
'''.strip()

model1 = 'gpt-5.4-mini'

rag_assistant1 = ev.RAGWithUsage(
    index=index,
    llm_client = openai_client,
    instructions = instruction1,
    prompt_template = ptemplate1,
    model = model1
)

In [5]:
llm_answer1 = rag_assistant1.rag(q['question'])
total_cost1 = rag_assistant1.total_cost()
print(llm_answer1)
print(f"Total cost:{total_cost1}")

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Yes. The context says tool wear becomes a concern once cumulative tool wear reaches a **critical band of roughly 200–240 minutes of continuous use**. In that window, the **failure probability rises sharply**.

**What’s happening**
- The tool is entering a wear-out phase, and failure can occur even if other signals still look normal.

**Why**
- The root cause is **progressive mechanical degradation of the cutting tool edge**.
- This is a **time/usage-driven failure mode**, not a stress-overload failure.
- The context explicitly notes that **torque, temperature, and rotational speed may remain nominal right up to failure**.

**Recommended action**
- **Flag any unit with tool wear > 200 minutes for priority inspection.**
- **Schedule tool replacement before 240 minutes** if it will be running for a long continuous period.
- **Do not rely o

In [6]:
def generate_rag_answer(q):
    question = q['question']
    doc_id = q['document']
    original_document = doc_idx[doc_id]

    llm_answer = rag_assistant1.rag(question)
    original_answer = original_document['answer']

    result = {
        'question':question,
        'llm_answer': llm_answer,
        'original_answer':original_answer,
        'document':doc_id
    }

    return result

In [7]:
record1 = generate_rag_answer(q)
record1

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71


{'question': 'Is there a typical time window where tool wear failure becomes likely during continuous use?',
 'llm_answer': 'Yes. The context says tool wear becomes a likely failure mode during a **critical band of about 200 to 240 minutes of continuous use**. In that window, the failure probability rises sharply, and it can happen even when **torque, temperature, and rotational speed still look normal**.\n\n### What’s happening\n- The tool’s wear accumulates over time.\n- Once cumulative tool wear reaches roughly **200–240 minutes**, the tool may fail without other obvious sensor anomalies.\n\n### Why\n- The root cause is **progressive mechanical degradation of the cutting tool edge**.\n- This is a **wear-out, time/usage-driven failure**, not a stress-overload failure.\n\n### Recommended action\n- **Flag any unit with tool wear > 200 minutes for priority inspection.**\n- **Schedule tool replacement before 240 minutes** if the unit is expected to run for a long time.\n- **Do not rely o

In [8]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = ev.map_progress(pool,dic_ground_truth,generate_rag_answer)

  0%|          | 0/355 [00:00<?, ?it/s]

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenanc

In [9]:
df_results =pd.DataFrame(results)
rag_assistant1.total_cost()


0.5672602500000001

In [10]:
df_results.head(5)

,question,llm_answer,original_answer,document
0,How can I tell if a tool is approaching wear f...,What’s happening: \n- This sounds like **Tool...,Tool wear time accumulates into a critical ban...,299b2d2994
1,Is there a typical time window where tool wear...,Yes. The context says tool wear failure become...,Tool wear time accumulates into a critical ban...,299b2d2994
2,Can tool wear failure happen without any obvio...,Yes.\n\n### What’s happening\nTool Wear Failur...,Tool wear time accumulates into a critical ban...,299b2d2994
3,What signs should I watch for when a tool has ...,When a tool has been running for around **200 ...,Tool wear time accumulates into a critical ban...,299b2d2994
4,Does tool wear failure usually show up as a su...,Yes. Tool Wear Failure can appear as a sudden ...,Tool wear time accumulates into a critical ban...,299b2d2994


In [11]:


# Save to rag answers file
df_results.to_csv(RAG_ANSWER_PATH / 'rag_answer.csv',index= False)

## 1.2 Preparing RAG Answer File 2

In [12]:
openai_client = OpenAI()
instruction2 = '''
You are a predictive maintenance assistant for industrial equipment.
Answer the question using ONLY the context below. If the context doesn't
contain enough information, say so rather than guessing.
'''
ptemplate2 = '''
QUESTION: {question}

CONTEXT:
{context}

Give a clear, short answer: what's happening, why , and recommended action.
'''.strip()

model2 = 'gpt-4o-mini'

rag_assistant2 = ev.RAGWithUsage(
    index=index,
    llm_client = openai_client,
    instructions = instruction2,
    prompt_template = ptemplate2,
    model = model2
)

In [13]:
llm_answer2 = rag_assistant2.rag(q['question'])
total_cost2 = rag_assistant2.total_cost()
print(llm_answer2)
print(f"Total cost:{total_cost2}")

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Tool wear failure becomes likely after approximately 200 to 240 minutes of continuous use due to the accumulation of mechanical degradation in the cutting tool edge. As wear increases, the probability of failure sharply rises, even if other indicators like torque and temperature appear normal. 

**Recommended action:** Flag any unit with tool wear exceeding 200 minutes for priority inspection and schedule tool replacement before reaching 240 minutes if the unit is queued for a long run.
Total cost:0.0008895000000000001


In [14]:
def generate_rag_answer2(q):
    question = q['question']
    doc_id = q['document']
    original_document = doc_idx[doc_id]

    llm_answer2 = rag_assistant2.rag(question)
    original_answer = original_document['answer']

    result = {
        'question':question,
        'llm_answer': llm_answer2,
        'original_answer':original_answer,
        'document':doc_id
    }

    return result

In [15]:
record2 = generate_rag_answer2(q)
record2

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71


{'question': 'Is there a typical time window where tool wear failure becomes likely during continuous use?',
 'llm_answer': 'Tool wear failure becomes likely between 200 and 240 minutes of continuous use due to the progressive mechanical degradation of the cutting tool edge. Once tool wear accumulates into this critical band, the probability of failure increases sharply, even if other sensor readings appear normal. \n\n**Recommended action:** Flag any unit with tool wear over 200 minutes for priority inspection and schedule tool replacement before reaching 240 minutes if the unit is queued for a long run.',
 'original_answer': 'Tool wear time accumulates into a critical band, typically between 200 and 240 minutes of continuous use. Within this band, failure probability rises sharply and can occur without any other sensor anomaly — torque, temperature, and rotational speed may all look nominal right up to the failure event.',
 'document': '299b2d2994'}

In [16]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results2 = ev.map_progress(pool,dic_ground_truth,generate_rag_answer2)

  0%|          | 0/355 [00:00<?, ?it/s]

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71
Loaded machine data:71
Loaded machine data:71
Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenanc

In [17]:
df_results2 =pd.DataFrame(results2)
rag_assistant2.total_cost()

0.30622200000000005

In [18]:
df_results2.head(5)

,question,llm_answer,original_answer,document
0,How can I tell if a tool is approaching wear f...,To determine if a tool is approaching wear fai...,Tool wear time accumulates into a critical ban...,299b2d2994
1,Is there a typical time window where tool wear...,Tool wear failure becomes likely after approxi...,Tool wear time accumulates into a critical ban...,299b2d2994
2,Can tool wear failure happen without any obvio...,"Yes, tool wear failure can occur without any o...",Tool wear time accumulates into a critical ban...,299b2d2994
3,What signs should I watch for when a tool has ...,Watch for increasing tool wear as the tool app...,Tool wear time accumulates into a critical ban...,299b2d2994
4,Does tool wear failure usually show up as a su...,"Yes, tool wear failure typically shows up as a...",Tool wear time accumulates into a critical ban...,299b2d2994


In [19]:
# Save to rag answers file
df_results2.to_csv(RAG_ANSWER_PATH / 'rag_answer2.csv',index= False)